# DWD Monthly Temperature Data to a Regional Time Series

This notebook adapts the DWD gridded climate data workflow to monthly mean air temperature.

It downloads monthly mean air temperature grids from the German Weather Service (DWD), converts them to GeoTIFF, clips them to the Kerpen study area, calculates a regional monthly mean, and exports the result as CSV and NetCDF.

Compared with the soil moisture workflow, the main dataset-specific changes are the DWD URL, filename prefix, value column, unit, and scale factor.


## Notebook environment

In [1]:
# Uncomment to install required Python packages
# !pip install -q numpy pandas rasterio geopandas xarray rioxarray netcdf4

In [2]:
from pathlib import Path
import os
import glob
import re
import subprocess

import rasterio
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray as rxr

## User settings

This section contains the dataset-specific settings for the temperature workflow.


### Adapted dataset: monthly mean air temperature

This notebook uses DWD monthly mean air temperature data:

```python
DWD_DATASET_FOLDER = "air_temperature_mean"
VALUE_COLUMN = "air_temperature_mean_celsius"
VALUE_UNIT = "°C"
```

The raw DWD raster values are stored in tenths of degrees Celsius. Therefore, values are multiplied by `0.1` before calculating the exported regional mean and before writing the NetCDF file.


In [26]:
# ---------------------------------------------------------------------
# USER SETTINGS
# ---------------------------------------------------------------------
# Change these values to adapt the workflow.
# ---------------------------------------------------------------------

REGION_NAME = "Kerpen"
START_YEAR = 1881
END_YEAR = 2026

# DWD dataset folder.
DWD_DATASET_FOLDER = "air_temperature_mean"

# Human-readable metadata for tables and plots.
DATASET_LABEL = "Monthly Mean Air Temperature"
VALUE_COLUMN_RAW = "air_temperature_mean_tenth_celsius"
VALUE_COLUMN = "air_temperature_mean_celsius"
VALUE_UNIT_RAW = "0.1 °C"
VALUE_UNIT = "°C"
SCALE_FACTOR = 0.1  # DWD temperature rasters are stored in tenths of °C.

# Coordinate reference system assigned during conversion from ASC to GeoTIFF.
# This should match the CRS of the DWD grid and the study-area shapefile.
SOURCE_CRS = "EPSG:31467"

# Main working directory.
# In Google Colab this is usually /content.
WORK_DIR = Path("/content")

# DWD source URL.
DWD_URL = (
    "https://opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/"
    f"{DWD_DATASET_FOLDER}/"
)

# Local folder for downloaded DWD files.
# The download command uses -nd, so all files are saved directly in this folder
# instead of preserving DWD's monthly subfolder structure.
DWD_DATA_DIR = WORK_DIR / DWD_DATASET_FOLDER

# Study-area shapefile.
# Update this path if your shapefile is stored somewhere else.
SHAPEFILE = (
    WORK_DIR
    / "shapefiles"
    / "kerpen_adm_25832"
    / "kerpen_adm_25832.shp"
)

# Output folders and files.
CROPPED_DIR = WORK_DIR / "cropped_temperature"
OUTPUT_CSV = WORK_DIR / f"{VALUE_COLUMN}_{REGION_NAME}_{START_YEAR}_{END_YEAR}.csv"
OUTPUT_NETCDF = WORK_DIR / f"{VALUE_COLUMN}_{REGION_NAME}_{START_YEAR}_{END_YEAR}.nc"

print("Dataset URL:", DWD_URL)
print("Local DWD folder:", DWD_DATA_DIR)
print("Shapefile:", SHAPEFILE)
print("Cropped folder:", CROPPED_DIR)
print("Output CSV:", OUTPUT_CSV)
print("Output NetCDF:", OUTPUT_NETCDF)


Dataset URL: https://opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/air_temperature_mean/
Local DWD folder: /content/air_temperature_mean
Shapefile: /content/shapefiles/kerpen_adm_25832/kerpen_adm_25832.shp
Cropped folder: /content/cropped_temperature
Output CSV: /content/air_temperature_mean_celsius_Kerpen_1881_2026.csv
Output NetCDF: /content/air_temperature_mean_celsius_Kerpen_1881_2026.nc


In [23]:
import shutil

shutil.unpack_archive("/content/shapefiles/kerpen_adm_25832/kerpen_adm_25832.zip")

In [4]:
# Pass Python variables to the shell environment.
# This makes them usable inside later `!` shell commands in Colab.
os.environ["DWD_DATA_DIR"] = str(DWD_DATA_DIR)
os.environ["SOURCE_CRS"] = str(SOURCE_CRS)
os.environ["DWD_URL"] = str(DWD_URL)

print("DWD_DATA_DIR:", os.environ["DWD_DATA_DIR"])
print("SOURCE_CRS:", os.environ["SOURCE_CRS"])
print("DWD_URL:", os.environ["DWD_URL"])


DWD_DATA_DIR: /content/air_temperature_mean
SOURCE_CRS: EPSG:31467
DWD_URL: https://opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/air_temperature_mean/


## 1. Download the DWD files

In [5]:
DWD_DATA_DIR.mkdir(parents=True, exist_ok=True)

!wget -r -np -nd -A "*.asc.gz" -P "$DWD_DATA_DIR" "$DWD_URL"


Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
Length: 224832 (220K) [application/octet-stream]
Saving to: ‘/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_196109.asc.gz’

grids_germany_month 100%[===================>] 219.56K  --.-KB/s    in 0.05s   

2026-07-09 15:56:17 (4.05 MB/s) - ‘/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_196109.asc.gz’ saved [224832/224832]

--2026-07-09 15:56:17--  https://opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/air_temperature_mean/09_Sep/grids_germany_monthly_air_temp_mean_196209.asc.gz
Reusing existing connection to opendata.dwd.de:443.
HTTP request sent, awaiting response... 200 OK
Length: 223074 (218K) [application/octet-stream]
Saving to: ‘/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_196209.asc.gz’

grids_germany_month 100%[===================>] 217.85K  --.-KB/s    in 0.05s   

2026-07-09 15:56:18 (4.07 MB/s) - ‘/content/air_temperature_mean/grids_ger

## 2. Check the downloaded files

In [6]:
file_count = 0

for root, dirs, files in os.walk(DWD_DATA_DIR):
    file_count += len(files)

print(f"Files downloaded: {file_count}")

Files downloaded: 1746


## 3. Inspect downloaded files and decompress the downloaded raster files

In [7]:
## Inspect downloaded files

!find "{DWD_DATA_DIR}" -type f

/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_200112.asc.gz
/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_199306.asc.gz
/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_200306.asc.gz
/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_196103.asc.gz
/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_202504.asc.gz
/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_190209.asc.gz
/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_199705.asc.gz
/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_191401.asc.gz
/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_191608.asc.gz
/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_201407.asc.gz
/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_201405.asc.gz
/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_202005.asc.gz
/content/air_temperature_mean/grids_germ

In [8]:
!find "{DWD_DATA_DIR}" -name "*.gz" -exec gunzip -f {{}} \;

In [9]:
# we can inspect the first 10 files, notice how ".asc.gz" is now ".asc"
os.listdir(DWD_DATA_DIR)[:10]

['grids_germany_monthly_air_temp_mean_202209.asc',
 'grids_germany_monthly_air_temp_mean_194312.asc',
 'grids_germany_monthly_air_temp_mean_192501.asc',
 'grids_germany_monthly_air_temp_mean_190509.asc',
 'grids_germany_monthly_air_temp_mean_193908.asc',
 'grids_germany_monthly_air_temp_mean_197405.asc',
 'grids_germany_monthly_air_temp_mean_202502.asc',
 'grids_germany_monthly_air_temp_mean_200203.asc',
 'grids_germany_monthly_air_temp_mean_190608.asc',
 'grids_germany_monthly_air_temp_mean_189511.asc']

## 4. Install GDAL

In [10]:
!apt-get install -y gdal-bin

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  python3-gdal python3-numpy
Suggested packages:
  libgdal-grass python-numpy-doc python3-dev python3-pytest
The following NEW packages will be installed:
  gdal-bin python3-gdal python3-numpy
0 upgraded, 3 newly installed, 0 to remove and 3 not upgraded.
Need to get 5,168 kB of archives.
After this operation, 25.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-numpy amd64 1:1.21.5-1ubuntu22.04.1 [3,467 kB]
Get:2 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy/main amd64 python3-gdal amd64 3.8.4+dfsg-1~jammy0 [1,095 kB]
Get:3 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy/main amd64 gdal-bin amd64 3.8.4+dfsg-1~jammy0 [605 kB]
Fetched 5,168 kB in 2s (3,359 kB/s)
Selecting previously unselected package python3-numpy.
(Reading database ... 118243 file

## 5. Check CRS and convert ASCII grids to GeoTIFF

In [11]:
# Let's check the metadata for the first .asc file
!gdalinfo "$(find "{DWD_DATA_DIR}" -name "*.asc" | head -1)"

Driver: AAIGrid/Arc/Info ASCII Grid
Files: /content/air_temperature_mean/grids_germany_monthly_air_temp_mean_202209.asc
Size is 654, 866
Origin = (3280414.711633467115462,6103500.628906250000000)
Pixel Size = (1000.000000000000000,-1000.000000000000000)
Corner Coordinates:
Upper Left  ( 3280414.712, 6103500.629) 
Lower Left  ( 3280414.712, 5237500.629) 
Upper Right ( 3934414.712, 6103500.629) 
Lower Right ( 3934414.712, 5237500.629) 
Center      ( 3607414.712, 5670500.629) 
Band 1 Block=654x1 Type=Int32, ColorInterp=Undefined
  NoData Value=-999


In [12]:
# We defined SOURCE_CRS as "EPSG:31467" in the settings at the beginning of the notebook.
# Here we use the shell environment variables created above.

!for f in $(find "$DWD_DATA_DIR" -name "*.asc"); do \
    gdal_translate -q -a_srs "$SOURCE_CRS" "$f" "${f%.asc}.tif"; \
done

# This command converts all decompressed DWD ASCII grid files to GeoTIFF
# while assigning the configured coordinate reference system.


In [13]:
# We can inspect the first 10 files in our folder again
os.listdir(DWD_DATA_DIR)[:10]

['grids_germany_monthly_air_temp_mean_193708.tif',
 'grids_germany_monthly_air_temp_mean_202209.asc',
 'grids_germany_monthly_air_temp_mean_189710.tif',
 'grids_germany_monthly_air_temp_mean_194312.asc',
 'grids_germany_monthly_air_temp_mean_190406.tif',
 'grids_germany_monthly_air_temp_mean_192501.asc',
 'grids_germany_monthly_air_temp_mean_190509.asc',
 'grids_germany_monthly_air_temp_mean_193908.asc',
 'grids_germany_monthly_air_temp_mean_197405.asc',
 'grids_germany_monthly_air_temp_mean_198310.tif']

In [14]:
# Let's have a closer look at what we got in our folder and files

# Check newly created GeoTIFF files
!find "$DWD_DATA_DIR" -name "*.tif" | head -5

/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_193708.tif
/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_189710.tif
/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_190406.tif
/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_198310.tif
/content/air_temperature_mean/grids_germany_monthly_air_temp_mean_190205.tif


In [15]:
# Count GeoTIFF files
!find "$DWD_DATA_DIR" -name "*.tif" | wc -l

1746


In [16]:
# Count GeoTIFF files
!find "$DWD_DATA_DIR" -name "*.asc" | wc -l

1746


In [17]:
# Inspect metadata and CRS of one converted GeoTIFF
!gdalinfo "$(find "$DWD_DATA_DIR" -name "*.tif" | head -1)" | head -80

Driver: GTiff/GeoTIFF
Files: /content/air_temperature_mean/grids_germany_monthly_air_temp_mean_193708.tif
Size is 654, 866
Coordinate System is:
PROJCRS["DHDN / 3-degree Gauss-Kruger zone 3",
    BASEGEOGCRS["DHDN",
        DATUM["Deutsches Hauptdreiecksnetz",
            ELLIPSOID["Bessel 1841",6377397.155,299.1528128,
                LENGTHUNIT["metre",1]]],
        PRIMEM["Greenwich",0,
            ANGLEUNIT["degree",0.0174532925199433]],
        ID["EPSG",4314]],
    CONVERSION["3-degree Gauss-Kruger zone 3",
        METHOD["Transverse Mercator",
            ID["EPSG",9807]],
        PARAMETER["Latitude of natural origin",0,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8801]],
        PARAMETER["Longitude of natural origin",9,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8802]],
        PARAMETER["Scale factor at natural origin",1,
            SCALEUNIT["unity",1],
            ID["EPSG",8805]],
        PARAMETER["False easti

In [18]:
# or get the CRS code using rasterio
# Expected CRS output should be something like:
# EPSG:31467

tif_files = glob.glob(os.path.join(os.environ["DWD_DATA_DIR"], "**", "*.tif"), recursive=True)

sample_tif = tif_files[0]

with rasterio.open(sample_tif) as src:
    print("Sample file:", sample_tif)
    print("CRS:", src.crs)

Sample file: /content/air_temperature_mean/grids_germany_monthly_air_temp_mean_193708.tif
CRS: EPSG:31467


### 6. Coordinate reference system

The DWD ASCII grids do not always store CRS metadata directly in the `.asc` files. Therefore, the CRS is assigned during conversion to GeoTIFF.

In this workflow, the DWD raster grid is assigned `EPSG:31467`, which corresponds to `DHDN / 3-degree Gauss-Krüger zone 3`.

The study-area shapefile must use the same CRS before clipping. If it does not, it is reprojected in the next section.


In [19]:
# Lastly, let's clean up the files we do not need anylonger, meaning the .asc files

# Delete decompressed ASCII grids after successful GeoTIFF conversion
!find "$DWD_DATA_DIR" -name "*.asc" -delete

In [20]:
# Verify that ASC files were removed
!find "$DWD_DATA_DIR" -name "*.asc" | wc -l

0


## 7. Reproject the study-area shapefile and clip the rasters to the study area

Before clipping, the study-area shapefile is reprojected to the same CRS as the DWD raster grid.

The resulting GeoTIFFs contain monthly mean air temperature values clipped to the Kerpen study area.


In [27]:
# Reproject shapefile to EPSG:31467 before clipping
# Read the study-area shapefile, we defined the path in the settings in the beginning of the notebook
study_area = gpd.read_file(SHAPEFILE)

print(f"Original shapefile CRS: {study_area.crs}")

Original shapefile CRS: EPSG:25832


In [28]:
# Reproject the shapefile to the same CRS as the DWD raster grid
study_area_31467 = study_area.to_crs(SOURCE_CRS)

REPROJECTED_SHAPEFILE = (
    SHAPEFILE.parent
    / f"{SHAPEFILE.stem}_{SOURCE_CRS.replace(':', '')}.shp"
)

study_area_31467.to_file(REPROJECTED_SHAPEFILE)

/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Field STAND created as String field, though DateTime requested.
  ogr_write(


In [29]:
# Check reprojected shapefile CRS
study_area_31467.crs

<Projected CRS: EPSG:31467>
Name: DHDN / 3-degree Gauss-Kruger zone 3
Axis Info [cartesian]:
- X[north]: Northing (metre)
- Y[east]: Easting (metre)
Area of Use:
- name: Germany - former West Germany onshore between 7°30'E and 10°30'E - states of Baden-Wurtemberg, Bayern, Bremen, Hamberg, Hessen, Niedersachsen, Nordrhein-Westfalen, Rhineland-Pfalz, Schleswig-Holstein.
- bounds: (7.5, 47.27, 10.51, 55.09)
Coordinate Operation:
- name: 3-degree Gauss-Kruger zone 3
- method: Transverse Mercator
Datum: Deutsches Hauptdreiecksnetz
- Ellipsoid: Bessel 1841
- Prime Meridian: Greenwich

In [30]:
CROPPED_DIR.mkdir(parents=True, exist_ok=True)

tif_files = list(DWD_DATA_DIR.rglob("*.tif"))

print(f"GeoTIFF files found: {len(tif_files)}")
print(f"Clipping shapefile: {REPROJECTED_SHAPEFILE}")

for tif_file in tif_files:
    output_name = tif_file.name.replace(
        "grids_germany_monthly_air_temp_mean_",
        "grids_cropped_monthly_air_temp_mean_",
    )

    output_file = CROPPED_DIR / output_name

    command = [
        "gdalwarp",
        "-q",
        "-cutline", str(REPROJECTED_SHAPEFILE),
        "-crop_to_cutline",
        "-dstalpha",
        str(tif_file),
        str(output_file),
    ]

    subprocess.run(command, check=True)

print(f"Clipped rasters saved to: {CROPPED_DIR}")


GeoTIFF files found: 1746
Clipping shapefile: /content/shapefiles/kerpen_adm_25832/kerpen_adm_25832_EPSG31467.shp
Clipped rasters saved to: /content/cropped_temperature


## 8. Calculate the regional monthly mean

This step converts the clipped monthly temperature rasters into a regional time series.

Each GeoTIFF represents one month, with the date encoded in the filename as `YYYYMM`, such as `199101` for January 1991.

The raw DWD temperature values are stored in tenths of degrees Celsius. Therefore, the regional mean is calculated from the valid pixels and then multiplied by `0.1` to convert the result to degrees Celsius.


In [31]:
# Collect clipped GeoTIFF files
files = sorted(CROPPED_DIR.glob("*.tif"))

print(f"Clipped GeoTIFF files found: {len(files)}")

records = []

for file_path in files:
    with rasterio.open(file_path) as src:
        # Read the first raster band as a masked array.
        # This automatically masks nodata values defined in the raster metadata.
        data = src.read(1, masked=True)

        # gdalwarp created an alpha band during clipping.
        # Pixels outside the study-area polygon have alpha = 0 and should be ignored.
        if src.count > 1:
            alpha = src.read(src.count)
            data = np.ma.masked_where(alpha == 0, data)

        # DWD temperature values are stored in tenths of degrees Celsius.
        regional_mean_raw = float(data.mean())
        regional_mean = regional_mean_raw * SCALE_FACTOR

    # Extract the date from the filename.
    # This assumes the filename contains the month as YYYYMM,
    # for example: grids_cropped_monthly_air_temp_mean_199101.tif
    #
    # If a different DWD dataset or filename pattern is used,
    # this regular expression may need to be adjusted.
    date_match = re.search(r"(\d{6})", file_path.name)

    if date_match is None:
        raise ValueError(f"No YYYYMM date found in filename: {file_path.name}")

    date_string = date_match.group(1)
    date = pd.to_datetime(date_string, format="%Y%m")

    records.append({
        "date": date,
        "filename": file_path.name,
        VALUE_COLUMN: regional_mean,
        "unit": VALUE_UNIT,
    })

df = (
    pd.DataFrame(records)
    .sort_values("date")
    .reset_index(drop=True)
)

df.head()


Clipped GeoTIFF files found: 1746


,date,filename,air_temperature_mean_celsius,unit
0,1881-01-01,grids_cropped_monthly_air_temp_mean_188101.tif,-2.693694,°C
1,1881-02-01,grids_cropped_monthly_air_temp_mean_188102.tif,3.192793,°C
2,1881-03-01,grids_cropped_monthly_air_temp_mean_188103.tif,5.554054,°C
3,1881-04-01,grids_cropped_monthly_air_temp_mean_188104.tif,8.082883,°C
4,1881-05-01,grids_cropped_monthly_air_temp_mean_188105.tif,13.606306,°C


## 9. Create the final time series table

The final table keeps only the date, value, and unit columns and sorts the records chronologically.

In [32]:
cleaned_df = (
    df[["date", VALUE_COLUMN, "unit"]]
    .copy()
    .sort_values(by="date")
    .reset_index(drop=True)
)

cleaned_df.head()

,date,air_temperature_mean_celsius,unit
0,1881-01-01,-2.693694,°C
1,1881-02-01,3.192793,°C
2,1881-03-01,5.554054,°C
3,1881-04-01,8.082883,°C
4,1881-05-01,13.606306,°C


In [33]:
cleaned_df.dtypes

,0
date,datetime64[ns]
air_temperature_mean_celsius,float64
unit,object


## 10. Export the results as CSV time series

The cleaned time series is saved as a CSV file and can be used for further analysis or visualization.


In [34]:
cleaned_df.to_csv(OUTPUT_CSV, index=False)

print("Saved to:", OUTPUT_CSV)
cleaned_df.head()

Saved to: /content/air_temperature_mean_celsius_Kerpen_1881_2026.csv


,date,air_temperature_mean_celsius,unit
0,1881-01-01,-2.693694,°C
1,1881-02-01,3.192793,°C
2,1881-03-01,5.554054,°C
3,1881-04-01,8.082883,°C
4,1881-05-01,13.606306,°C


## 11. Export GeoTIFFs to NetCDF

The clipped monthly GeoTIFF files can be combined into a single NetCDF file.

Each GeoTIFF represents one month. The date is extracted from the filename using the `YYYYMM` pattern. The rasters are then stacked along a new `time` dimension.

The raw DWD temperature values are converted from tenths of degrees Celsius to degrees Celsius before export.


In [35]:
netcdf_layers = []

clipped_files = sorted(CROPPED_DIR.glob("*.tif"))

print(f"Clipped GeoTIFF files found: {len(clipped_files)}")

for file_path in clipped_files:
    # Extract the date from the filename.
    # This assumes the filename contains the month as YYYYMM,
    # for example: grids_cropped_monthly_air_temp_mean_199101.tif
    #
    # If a different DWD dataset or filename pattern is used,
    # this regular expression may need to be adjusted.
    date_match = re.search(r"(\d{6})", file_path.name)

    if date_match is None:
        raise ValueError(f"No YYYYMM date found in filename: {file_path.name}")

    date_string = date_match.group(1)
    date = pd.to_datetime(date_string, format="%Y%m")

    # Open the raster as an xarray DataArray.
    # masked=True converts nodata pixels to NaN.
    raster = rxr.open_rasterio(file_path, masked=True)

    # Select the first data band.
    # The clipped GeoTIFFs may also contain an alpha band from gdalwarp.
    raster = raster.sel(band=1).drop_vars("band")

    # DWD temperature values are stored in tenths of degrees Celsius.
    # Convert them to degrees Celsius before exporting.
    raster = raster * SCALE_FACTOR

    # Add the month as a time coordinate.
    raster = raster.expand_dims(time=[date])

    netcdf_layers.append(raster)

temperature_cube = xr.concat(netcdf_layers, dim="time")

temperature_cube.name = VALUE_COLUMN
temperature_cube.attrs["units"] = VALUE_UNIT
temperature_cube.attrs["raw_units"] = VALUE_UNIT_RAW
temperature_cube.attrs["scale_factor_applied"] = SCALE_FACTOR
temperature_cube.attrs["long_name"] = "Monthly mean air temperature"
temperature_cube.attrs["description"] = (
    "Regional clipped DWD monthly mean air temperature rasters stacked along the time dimension. "
    "Raw values were converted from tenths of degrees Celsius to degrees Celsius."
)

temperature_cube


Clipped GeoTIFF files found: 1746


<xarray.DataArray 'air_temperature_mean_celsius' (time: 1746, y: 10, x: 17)> Size: 2MB
array([[[ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
        [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
        [-2.5, -2.5, -2.6, ...,  0. ,  0. ,  0. ],
        ...,
        [-2.6, -2.5, -2.7, ..., -2.8, -2.8, -2.8],
        [ 0. ,  0. ,  0. , ..., -2.7, -2.8,  0. ],
        [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ]],

       [[ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
        [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
        [ 3.3,  3.3,  3.3, ...,  0. ,  0. ,  0. ],
        ...,
        [ 3.2,  3.3,  3.2, ...,  3.1,  3.1,  3.1],
        [ 0. ,  0. ,  0. , ...,  3.2,  3.1,  0. ],
        [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ]],

       [[ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
        [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
        [ 5.6,  5.6,  5.6, ...,  0. ,  0. ,  0. ],
        ...,
...
        ...,
        [10.8, 10.8, 10.7, ..., 10.7, 10.7, 10.7],
        [ 0. ,  0. ,  0. , ..., 10.8, 10.7,  0. ],
        [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ]],

       [[ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
        [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
        [15.1, 15.1, 15.1, ...,  0. ,  0. ,  0. ],
        ...,
        [15.1, 15.1, 15.1, ..., 15. , 15. , 15. ],
        [ 0. ,  0. ,  0. , ..., 15.1, 15. ,  0. ],
        [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ]],

       [[ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
        [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
        [20.6, 20.6, 20.6, ...,  0. ,  0. ,  0. ],
        ...,
        [20.4, 20.5, 20.5, ..., 20.5, 20.5, 20.5],
        [ 0. ,  0. ,  0. , ..., 20.6, 20.5,  0. ],
        [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ]]])
Coordinates:
  * time         (time) datetime64[ns] 14kB 1881-01-01 1881-02-01 ... 2026-06-01
  * y            (y) float64 80B 5.645e+06 5.644e+06 ... 5.637e+06 5.636e+06
  * x            (x) float64 136B 3.329e+06 3.33e+06 ... 3.344e+06 3.345e+06
    spatial_ref  int64 8B 0
Attributes:
    AREA_OR_POINT:         Area
    scale_factor:          1.0
    add_offset:            0.0
    units:                 °C
    raw_units:             0.1 °C
    scale_factor_applied:  0.1
    long_name:             Monthly mean air temperature
    description:           Regional clipped DWD monthly mean air temperature ...

In [36]:
temperature_cube.to_netcdf(OUTPUT_NETCDF)

print(f"NetCDF saved to: {OUTPUT_NETCDF}")


NetCDF saved to: /content/air_temperature_mean_celsius_Kerpen_1881_2026.nc


In [37]:
# Let's have a look at the file we just created
ds_check = xr.open_dataset(OUTPUT_NETCDF)
ds_check

<xarray.Dataset> Size: 2MB
Dimensions:                       (time: 1746, y: 10, x: 17)
Coordinates:
  * time                          (time) datetime64[ns] 14kB 1881-01-01 ... 2...
  * y                             (y) float64 80B 5.645e+06 ... 5.636e+06
  * x                             (x) float64 136B 3.329e+06 ... 3.345e+06
    spatial_ref                   int64 8B ...
Data variables:
    air_temperature_mean_celsius  (time, y, x) float64 2MB ...

In [38]:
print("Start:", ds_check.time.min().values)
print("End:", ds_check.time.max().values)
print("Time steps:", ds_check.sizes["time"])

Start: 1881-01-01T00:00:00.000000000
End: 2026-06-01T00:00:00.000000000
Time steps: 1746
